# Chapter 22 — Convolutional Neural Networks II: ResNet, Training, Transfer Learning & Applications

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch22_convolutional_neural_networks_ii/ch22_notebook.ipynb)

Companion notebook for **Chapter 22** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Stacking many plain layers causes gradients to vanish or explode during backpropagation.
- Residual (skip) connections add the input to the output, giving gradients a direct path and enabling networks 50 to 152 layers deep.
- In backpropagation, max pooling sends the gradient only to the winning element, while mean pooling shares it equally.
- Transfer learning reuses a representation already learned on a large dataset, which is invaluable when labeled data is scarce.
- CNNs extend well beyond classification: U-Net segments images, YOLO and Faster R-CNN detect objects, and 1-D and 3-D CNNs handle signals and video.

## Example 1: Residual Connections vs Vanishing Gradients

This simulates backpropagation through a 20-layer network. In a plain network the gradient is multiplied by a small factor at each layer and vanishes to ~1e-19 by the first layer. With residual (skip) connections the gradient has a '+1' highway and stays healthy. This is exactly why ResNet can train networks 50-152 layers deep.

In [1]:
# Chapter 22 - Example 1: Residual connections fix vanishing gradients
import numpy as np
np.random.seed(0)

def sigmoid(x): return 1/(1+np.exp(-x))
def sigmoid_deriv(x): return sigmoid(x)*(1-sigmoid(x))

# Simulate gradient flow backward through a deep network
# Plain net: gradient multiplies by (weight * activation_deriv) each layer
# Residual net: gradient flows through identity skip (+1) so it doesn't vanish

n_layers = 20
np.random.seed(0)
weights = np.random.uniform(0.3, 0.9, n_layers)   # typical small weights
acts    = np.random.uniform(0.1, 0.25, n_layers)  # sigmoid derivative range

print("=== Gradient Flow Through a 20-Layer Network ===\n")
print("Plain network: grad_(L-1) = grad_L * w * sigmoid'(a)")
print("Residual net : grad_(L-1) = grad_L * (1 + w * sigmoid'(a))  <- the +1 is the skip\n")

grad_plain = 1.0
grad_res   = 1.0
print(f"{'Layer':>6} {'Plain grad':>14} {'Residual grad':>16}")
print("-" * 40)
for L in range(n_layers, 0, -1):
    factor = weights[L-1] * acts[L-1]
    grad_plain *= factor
    grad_res   *= (1 + factor)
    if L % 4 == 0 or L == 1:
        print(f"{L:>6} {grad_plain:>14.2e} {grad_res:>16.2e}")

print(f"\nAfter backprop through all 20 layers:")
print(f"  Plain network gradient   : {grad_plain:.2e}  (vanished to near-zero!)")
print(f"  Residual network gradient: {grad_res:.2e}  (healthy, training continues)")
print(f"\nRatio: residual gradient is {grad_res/grad_plain:.2e}x larger")
print("This is why ResNet can train 50, 101, even 152 layers deep.")

=== Gradient Flow Through a 20-Layer Network ===

Plain network: grad_(L-1) = grad_L * w * sigmoid'(a)
Residual net : grad_(L-1) = grad_L * (1 + w * sigmoid'(a))  <- the +1 is the skip

 Layer     Plain grad    Residual grad
----------------------------------------
    20       1.66e-01         1.17e+00
    16       1.93e-05         1.81e+00
    12       1.55e-09         2.72e+00
     8       4.56e-13         4.55e+00
     4       3.73e-17         6.67e+00
     1       1.04e-19         9.95e+00

After backprop through all 20 layers:
  Plain network gradient   : 1.04e-19  (vanished to near-zero!)
  Residual network gradient: 9.95e+00  (healthy, training continues)

Ratio: residual gradient is 9.58e+19x larger
This is why ResNet can train 50, 101, even 152 layers deep.


## Example 2: Backpropagation Through Pooling Layers

Pooling layers have no learnable weights, but gradients still must flow back through them. This example shows the fixed routing weights: max pooling sends the whole gradient only to the element that 'won' the max (a 0/1 mask), while mean pooling spreads the gradient equally (1/F^2) across the region. This is the rule used when training a CNN.

In [2]:
# Chapter 22 - Example 2: Max-pool vs Mean-pool BACKWARD pass (gradient routing)
import numpy as np

print("=== Backpropagation Through Pooling Layers ===\n")
print("Forward: pooling reduces a region to one value.")
print("Backward: the upstream gradient must be routed back to the input region.\n")

# A 2x2 region in the feature map
region = np.array([[0.8, -0.1],
                   [-1.3, 1.5]])
upstream_grad = 2.0  # gradient flowing back from next layer

print("Input region (2x2):")
print(region)
print(f"Upstream gradient arriving at this pooled node: {upstream_grad}\n")

# --- MAX POOLING backward ---
print("--- Max Pooling ---")
max_val = region.max()
mask = (region == max_val).astype(float)
print(f"Forward output (max): {max_val}")
print("Backward weights (1 only at the max location, 0 elsewhere):")
print(mask.astype(int))
grad_max = mask * upstream_grad
print("Gradient routed to input region:")
print(grad_max)
print("Only the neuron that 'won' the max receives the gradient.\n")

# --- MEAN POOLING backward ---
print("--- Mean Pooling ---")
mean_val = region.mean()
F2 = region.size  # F*F = 4
print(f"Forward output (mean): {mean_val:.3f}")
print(f"Backward weights (1/F^2 = 1/{F2} = {1/F2} at every location):")
print(np.full(region.shape, 1/F2))
grad_mean = np.full(region.shape, 1/F2) * upstream_grad
print("Gradient routed to input region:")
print(grad_mean)
print("Every neuron in the region shares the gradient equally.\n")

print("=== Summary ===")
print("Max pooling : gradient goes ONLY to the max element (sparse, 0/1 weights)")
print("Mean pooling: gradient is SPREAD evenly (each gets 1/F^2)")
print("Pooling layers have no learnable weights - these routing weights are fixed.")

=== Backpropagation Through Pooling Layers ===

Forward: pooling reduces a region to one value.
Backward: the upstream gradient must be routed back to the input region.

Input region (2x2):
[[ 0.8 -0.1]
 [-1.3  1.5]]
Upstream gradient arriving at this pooled node: 2.0

--- Max Pooling ---
Forward output (max): 1.5
Backward weights (1 only at the max location, 0 elsewhere):
[[0 0]
 [0 1]]
Gradient routed to input region:
[[0. 0.]
 [0. 2.]]
Only the neuron that 'won' the max receives the gradient.

--- Mean Pooling ---
Forward output (mean): 0.225
Backward weights (1/F^2 = 1/4 = 0.25 at every location):
[[0.25 0.25]
 [0.25 0.25]]
Gradient routed to input region:
[[0.5 0.5]
 [0.5 0.5]]
Every neuron in the region shares the gradient equally.

=== Summary ===
Max pooling : gradient goes ONLY to the max element (sparse, 0/1 weights)
Mean pooling: gradient is SPREAD evenly (each gets 1/F^2)
Pooling layers have no learnable weights - these routing weights are fixed.


## Example 3: Transfer Learning (Few-Shot)

This shows transfer learning. We pretrain a deep model on a large set of digit classes, then tackle two brand-new classes with only 10 labeled images. Reusing the pretrained features and training a tiny classifier on top beats training a fresh model from scratch. This mirrors how a VGGNet pretrained on ImageNet is reused for medical images with little labeled data.

In [3]:
# Chapter 22 - Example 3: Transfer learning simulation (few-shot target domain)
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(0)

print("=== Transfer Learning: Reusing a Pretrained Representation ===\n")

digits = load_digits()
X, y = digits.data, digits.target

# Source domain: a LARGE labeled dataset (8 of the 10 digit classes)
src_classes = [0,1,2,3,4,5,6,7]
src = np.isin(y, src_classes)
Xs, ys = X[src], y[src]
print(f"Source domain : digits {src_classes}")
print(f"                {len(Xs)} labeled images -> pretrain a deep CNN-like model")
pretrained = MLPClassifier(hidden_layer_sizes=(120, 64), max_iter=600, random_state=0)
pretrained.fit(Xs, ys)
print(f"                pretrained source accuracy: {pretrained.score(Xs, ys)*100:.1f}%\n")

# Target domain: NEW classes (8, 9) with only a handful of labels each
tgt = np.isin(y, [8, 9])
Xt, yt = X[tgt], y[tgt]
# Only 5 examples per class = 10 total labeled target images (few-shot)
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    Xt, yt, train_size=10, random_state=0, stratify=yt)
print(f"Target domain : NEW digits 8 and 9")
print(f"                only {len(Xt_train)} labeled images (5 per class - few-shot!)\n")

# Pretrained feature extractor (hidden activations)
def extract_features(model, Xin):
    a = Xin
    for W, b in zip(model.coefs_[:-1], model.intercepts_[:-1]):
        a = np.maximum(0, a @ W + b)
    return a

ft_tr = extract_features(pretrained, Xt_train)
ft_te = extract_features(pretrained, Xt_test)

# Approach 1: transfer - simple classifier on pretrained features
transfer = LogisticRegression(max_iter=2000, random_state=0)
transfer.fit(ft_tr, yt_train)
acc_transfer = accuracy_score(yt_test, transfer.predict(ft_te))

# Approach 2: from scratch on raw pixels, same 10 examples
scratch = MLPClassifier(hidden_layer_sizes=(120, 64), max_iter=600, random_state=0)
scratch.fit(Xt_train, yt_train)
acc_scratch = accuracy_score(yt_test, scratch.predict(Xt_test))

print(f"Test set: {len(yt_test)} unseen images of digits 8 and 9\n")
print("Results:")
print(f"  Train from scratch (raw pixels, 10 imgs) : {acc_scratch*100:.1f}%")
print(f"  Transfer learning (pretrained features)  : {acc_transfer*100:.1f}%")
print(f"\nImprovement from transfer learning: {(acc_transfer-acc_scratch)*100:+.1f} pp")
print("\nWith only 10 labeled images, training a deep model from scratch fails to")
print("generalize. The pretrained model already learned reusable stroke/edge")
print("features, so a tiny classifier on top of them generalizes far better.")
print("This is exactly how a VGGNet pretrained on ImageNet is reused for")
print("medical images with only a few labeled examples.")

=== Transfer Learning: Reusing a Pretrained Representation ===

Source domain : digits [0, 1, 2, 3, 4, 5, 6, 7]
                1443 labeled images -> pretrain a deep CNN-like model
                pretrained source accuracy: 100.0%

Target domain : NEW digits 8 and 9
                only 10 labeled images (5 per class - few-shot!)

Test set: 344 unseen images of digits 8 and 9

Results:
  Train from scratch (raw pixels, 10 imgs) : 70.3%
  Transfer learning (pretrained features)  : 77.0%

Improvement from transfer learning: +6.7 pp

With only 10 labeled images, training a deep model from scratch fails to
generalize. The pretrained model already learned reusable stroke/edge
features, so a tiny classifier on top of them generalizes far better.
This is exactly how a VGGNet pretrained on ImageNet is reused for
medical images with only a few labeled examples.
